# `demo_v5`: OT maps, bootstrap bands, and slide-vs-bootstrap variance for v5

Same idea as `Demo.ipynb`/`Demo_mini.ipynb` (built against `profiles_v3.hdf5`), redone for
v5 -- no fits, just plots + error quantification. Uses the 4 offset slides built in
`slides_v5.ipynb`: `bootstrap_v5.csv` (off=0.0) and `bootstrap_slide_v5_{0.03125,0.0625,
0.09375}.csv`. Each already has `rc`, `ot`, `ot_std_boot` per (bin, radius), so loading is
simpler than the v3 version (no separate rank-based CSV merge needed).

1. Plot the slides (all 4 offsets overlaid, one mass bin) and neighboring-mass-bin grids, both
   with **bootstrap $\pm1\sigma$ shaded bands**.
2. **Quantify variance in the bins vs bootstrap**: `bin_and_total_error` interpolates each of
   the 4 offsets' nearest-bin curve onto a common grid; the std *across* those 4 curves at each
   radius is the binning-choice sigma, compared against the bootstrap sigma from the primary
   (off=0) slide, combined in quadrature.
3. **Smoothness in (log M, z)**: the same binning-vs-bootstrap ratio, computed over every mass
   bin and snapshot for a sim and plotted as a 2D map -- ratio >> 1 means the OT map's shape is
   more sensitive to where you draw the bin edges than to pure halo-count noise (not smooth);
   ratio ~< 1 means bootstrap noise dominates (smooth, bin-placement-robust).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 16, "axes.titlesize": 18, "axes.labelsize": 16,
    "legend.fontsize": 13, "xtick.labelsize": 13, "ytick.labelsize": 13,
    "figure.titlesize": 20, "figure.dpi": 110,
})

PATH = "/Users/jpaine/Desktop/Flamingo/"
OFFS = [0, .03125, .0625, .09375]
SNAP_Z = {9:5.,13:4.,17:3.,37:2.,47:1.5,57:1.,62:.75,67:.5,69:.4,71:.3,73:.2,75:.1,77:0.}
W = 0.125

In [ ]:
# off=0 is profiles_v5_10k.hdf5's own bootstrap; the other 3 are the rebinned slides from
# slides_v5.ipynb. Each CSV already has rc/ot/ot_std_boot per (bin, radius) -- just derive
# ot_disp = ot - rc (the displacement people actually plot) and rename for clarity.
BOOT_FILES = {
    0:      f"{PATH}bootstrap_v5.csv",
    .03125: f"{PATH}bootstrap_slide_v5_0.03125.csv",
    .0625:  f"{PATH}bootstrap_slide_v5_0.06250.csv",
    .09375: f"{PATH}bootstrap_slide_v5_0.09375.csv",
}

SLIDES = {}
for off, fn in BOOT_FILES.items():
    d = pd.read_csv(fn, dtype={"snap": str})
    d["ot_disp"] = d["ot"] - d["rc"]
    d = d.rename(columns={"ot_std_boot": "ot_disp_std"})
    SLIDES[off] = d
    print(f"off={off:.5f}: {len(d)} rows, "
          f"{d[['snap','sim','lo','hi']].drop_duplicates().shape[0]} bins")

In [ ]:
# single bin, with its bootstrap +-1sigma band
def plot_bin(sim, snap, off, mass):
    d = SLIDES[off]; d = d[(d.snap == snap) & (d.sim == sim)]
    lo = d.lo.iloc[(d.lo - mass).abs().values.argmin()]
    b = d[np.isclose(d.lo, lo)].sort_values("rc")
    plt.fill_between(b.rc, b.ot_disp - b.ot_disp_std, b.ot_disp + b.ot_disp_std,
                      alpha=.3, label=r"$\pm1\sigma$ bootstrap")
    plt.plot(b.rc, b.ot_disp, "-", label="OT displacement")
    plt.xscale("log"); plt.axhline(0, color="gray", lw=.5)
    plt.legend()
    plt.xlabel("r / R200c"); plt.ylabel(r"$T(r)-r$")
    plt.title(f"{sim} snap={snap} off={off:.5f} logM=[{lo:.3f},{lo+W:.3f})")
    plt.show()

plot_bin("Jet", "77", OFFS[0], 13.5)

In [ ]:
# the 4 slides overlaid for one (sim, snap, mass), each with its own bootstrap band
def plot_compare_slides(sim, snap, mass, offs=OFFS):
    fig, ax = plt.subplots(figsize=(7, 5))
    for j, off in enumerate(offs):
        d = SLIDES[off]; d = d[(d.snap == snap) & (d.sim == sim)]
        if d.empty:
            continue
        lo = d.lo.iloc[(d.lo - mass).abs().values.argmin()]
        b = d[np.isclose(d.lo, lo)].sort_values("rc")
        c = plt.cm.viridis(j / max(len(offs) - 1, 1))
        ax.fill_between(b.rc, b.ot_disp - b.ot_disp_std, b.ot_disp + b.ot_disp_std, color=c, alpha=.2)
        ax.plot(b.rc, b.ot_disp, "-", color=c, label=f"off={off:.5f} logM=[{lo:.3f},{lo+W:.3f})")
    ax.set_xscale("log"); ax.axhline(0, color="gray", lw=.5)
    ax.legend(fontsize=11)
    ax.set_xlabel("r / R200c"); ax.set_ylabel(r"$T(r)-r$")
    ax.set_title(f"{sim} snap={snap}: {len(offs)} slides compared, with " r"$\pm1\sigma$ bootstrap bands")
    plt.show()

plot_compare_slides("Jet", "77", 13.5)
plot_compare_slides("fgas-2sigma", "77", 12.5)

In [ ]:
# a grid of neighboring mass bins, one offset at a time, each with its bootstrap band
def plot_mass_neighbors(sim, snap, off, n=5, mass=None, end="low"):
    off_idx = OFFS.index(off) if off in OFFS else int(np.argmin(np.abs(np.array(OFFS) - off)))
    c = plt.cm.viridis(off_idx / max(len(OFFS) - 1, 1))

    d = SLIDES[off]; d = d[(d.snap == snap) & (d.sim == sim)]
    los_all = np.array(sorted(d.lo.unique()))
    if mass is not None:
        idx = int(np.argmin(np.abs(los_all - mass)))
        los = los_all[idx:idx + n] if end == "high" else los_all[max(0, idx - n + 1):idx + 1]
    else:
        los = los_all[-n:] if end == "high" else los_all[:n]

    fig, ax = plt.subplots(1, len(los), figsize=(4.2 * len(los), 3.8), sharey=True)
    ax = np.atleast_1d(ax)
    for j, lo in enumerate(los):
        b = d[np.isclose(d.lo, lo)].sort_values("rc")
        ax[j].fill_between(b.rc, b.ot_disp - b.ot_disp_std, b.ot_disp + b.ot_disp_std, color=c, alpha=.3)
        ax[j].plot(b.rc, b.ot_disp, "-", color=c)
        ax[j].set_xscale("log"); ax[j].axhline(0, color="gray", lw=.5)
        ax[j].set_title(f"logM=[{lo:.3f},{lo+W:.3f})", fontsize=12)
    ax[0].set_ylabel(r"$T(r)-r$")
    plt.suptitle(f"{sim} snap={snap} off={off:.5f}: {len(los)} neighboring mass bins "
                 r"($\pm1\sigma$ bootstrap)", y=1.03)
    plt.tight_layout(); plt.show()

for off in OFFS:
    plot_mass_neighbors("Jet", "77", off, n=8, end="low")

## Quantifying variance: bootstrap error vs. binning (slide) error

For a target `(sim, snap, mass)`, each of the 4 offsets contributes its own nearest bin's
curve. Interpolated onto a common grid, the **std across those 4 curves** at each radius is
the binning-choice sigma (`bin_std`) -- how much the answer would've changed had the mass grid
been drawn slightly differently. The **bootstrap sigma** (`boot_std`) from the primary (off=0)
slide is the pure halo-count statistical noise. Combined in quadrature: `total_std =
sqrt(boot_std^2 + bin_std^2)`.

In [ ]:
def bin_and_total_error(sim, snap, mass, offs=OFFS):
    grid = np.geomspace(0.15, 4.85, 100)   # same common grid used elsewhere in this project
    curves = []
    for off in offs:
        d = SLIDES[off]; d = d[(d.snap == snap) & (d.sim == sim)]
        if d.empty:
            continue
        lo = d.lo.iloc[(d.lo - mass).abs().values.argmin()]
        b = d[np.isclose(d.lo, lo)].sort_values("rc")
        curves.append(np.interp(np.log(grid), np.log(b.rc), b.ot_disp))
    bin_std = np.array(curves).std(axis=0)

    d0 = SLIDES[0]; d0 = d0[(d0.snap == snap) & (d0.sim == sim)]
    lo0 = d0.lo.iloc[(d0.lo - mass).abs().values.argmin()]
    b0 = d0[np.isclose(d0.lo, lo0)].sort_values("rc")
    y = np.interp(np.log(grid), np.log(b0.rc), b0.ot_disp)
    boot_std = np.interp(np.log(grid), np.log(b0.rc), b0.ot_disp_std)
    total_std = np.sqrt(boot_std**2 + bin_std**2)
    return grid, y, boot_std, bin_std, total_std

def plot_bin_total_error(sim, snap, mass):
    grid, y, boot_std, bin_std, total_std = bin_and_total_error(sim, snap, mass)
    plt.fill_between(grid, y - total_std, y + total_std, alpha=.25, label="total (bootstrap+binning)")
    plt.fill_between(grid, y - boot_std, y + boot_std, alpha=.4, label="bootstrap only")
    plt.plot(grid, y, "k-", label="OT displacement (off=0)")
    plt.xscale("log"); plt.axhline(0, color="gray", lw=.5)
    plt.legend()
    z = SNAP_Z.get(int(snap), np.nan)
    plt.title(f"{sim} snap={snap} (z={z}) logM~{mass}: bootstrap-only vs total error")
    plt.show()
    ratio = np.nanmedian(bin_std) / np.nanmedian(boot_std) if np.nanmedian(boot_std) > 0 else np.nan
    print(f"median boot_std={np.nanmedian(boot_std):.4f}  bin_std={np.nanmedian(bin_std):.4f}  "
          f"total_std={np.nanmedian(total_std):.4f}  bin/boot ratio={ratio:.2f}")

plot_bin_total_error("Jet", "77", 13.5)
plot_bin_total_error("Jet", "77", 12.5)
plot_bin_total_error("Jet", "17", 12.5)

In [ ]:
plot_bin_total_error("fgas-2sigma", "77", 13.5)
plot_bin_total_error("fgas-2sigma", "77", 12.5)
plot_bin_total_error("fgas-2sigma", "17", 12.5)

## How smooth is the OT map in (log M, z)?

Same `bin_and_total_error` machinery, now swept over every mass bin and snapshot for one sim.
The ratio `median(bin_std) / median(boot_std)` at each (log M, z) point is the smoothness
diagnostic: **>1 means the map's shape depends more on where the bin edges happen to fall than
on pure halo-count noise** (not smooth / not robust to bin placement); **<~1 means bootstrap
noise dominates** (smooth, bin-placement-robust).

In [ ]:
def smoothness_map(sim):
    d0 = SLIDES[0]; d0 = d0[d0.sim == sim]
    rows = []
    for snap, g in d0.groupby("snap"):
        z = SNAP_Z.get(int(snap), np.nan)
        for lo in sorted(g.lo.unique()):
            mass = lo + W / 2
            try:
                _, _, boot_std, bin_std, _ = bin_and_total_error(sim, snap, mass)
            except Exception:
                continue
            mb, mo = np.nanmedian(bin_std), np.nanmedian(boot_std)
            rows.append(dict(snap=snap, z=z, logM=mass, med_boot=mo, med_bin=mb,
                              ratio=mb / mo if mo > 0 else np.nan))
    return pd.DataFrame(rows)

def plot_smoothness_map(sim):
    df = smoothness_map(sim)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    specs = [("med_boot", "median bootstrap sigma", "viridis", None),
             ("med_bin", "median binning sigma\n(std across the 4 slides)", "viridis", None),
             ("ratio", "binning / bootstrap ratio\n(>1: binning matters more than noise)", "coolwarm", (0, 2))]
    for ax, (col, title, cmap, vr) in zip(axes, specs):
        norm = plt.Normalize(*vr) if vr else plt.Normalize(0, np.nanpercentile(df[col], 95))
        sc = ax.scatter(df.logM, df.z, c=df[col], cmap=cmap, norm=norm, s=55, edgecolor="k", linewidth=.3)
        fig.colorbar(sc, ax=ax)
        ax.set_xlabel("logM"); ax.set_title(title, fontsize=12)
    axes[0].set_ylabel("z")
    fig.suptitle(f"{sim}: how smooth is the OT map in (logM, z)?", y=1.05)
    plt.tight_layout(); plt.show()
    print(f"median ratio overall: {df.ratio.median():.2f}  "
          f"({(df.ratio>1).mean():.0%} of (logM,z) points have ratio>1)")
    return df

df_smooth_jet = plot_smoothness_map("Jet")

In [ ]:
df_smooth_fgas4 = plot_smoothness_map("fgas-4sigma")